# Aero Predict — Baseline FD001

Ce notebook contient la baseline Random Forest utilisée pour prédire la durée de vie utile restante (RUL) des moteurs du dataset NASA C-MAPSS FD001.

Le pipeline retenu comprend :

* le chargement et la vérification des données ;
* le calcul de la RUL plafonnée à 125 cycles ;
* la suppression des variables constantes ;
* la création de caractéristiques temporelles sur 5 cycles : moyenne, écart-type et variation ;
* une validation croisée par moteur avec `GroupKFold` ;
* une Random Forest optimisée ;
* l’évaluation avec le RMSE et le score asymétrique NASA.

Configuration finale de la Random Forest :

n_estimators=100,
max_depth=12,
min_samples_leaf=3,
random_state=42

## 1. Importation des bibliothèques

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

## 2. Chargement des données

In [2]:
DATA_DIR = Path("data")

files = [
    "train_FD001.txt",
    "test_FD001.txt",
    "RUL_FD001.txt"
]

for filename in files:
    path = DATA_DIR / filename
    print(filename, ":", path.exists())

train_FD001.txt : True
test_FD001.txt : True
RUL_FD001.txt : True


In [3]:
column_names = (
    ["engine_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

test = pd.read_csv(
    DATA_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

rul_test = pd.read_csv(
    DATA_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["rul"]
)

print("Train :", train.shape)
print("Test  :", test.shape)
print("RUL   :", rul_test.shape)

Train : (20631, 26)
Test  : (13096, 26)
RUL   : (100, 1)


## 3. Vérification des données

In [4]:
print("Moteurs train :", train["engine_id"].nunique())
print("Moteurs test  :", test["engine_id"].nunique())

print("Valeurs manquantes train :", train.isna().sum().sum())
print("Valeurs manquantes test  :", test.isna().sum().sum())

print("Doublons train :", train.duplicated().sum())
print("Doublons test  :", test.duplicated().sum())

Moteurs train : 100
Moteurs test  : 100
Valeurs manquantes train : 0
Valeurs manquantes test  : 0
Doublons train : 0
Doublons test  : 0


## 4. Construction de la cible RUL

In [5]:
RUL_CAP = 125

train["max_cycle"] = (
    train.groupby("engine_id")["cycle"]
    .transform("max")
)

train["rul"] = (
    train["max_cycle"] - train["cycle"]
)

train["rul_capped"] = (
    train["rul"].clip(upper=RUL_CAP)
)

print("RUL minimale :", train["rul"].min())
print("RUL maximale :", train["rul"].max())

print(
    "RUL plafonnée minimale :",
    train["rul_capped"].min()
)

print(
    "RUL plafonnée maximale :",
    train["rul_capped"].max()
)

print(
    "Tous les moteurs terminent à zéro :",
    train.groupby("engine_id")
    .tail(1)["rul"]
    .eq(0)
    .all()
)

RUL minimale : 0
RUL maximale : 361
RUL plafonnée minimale : 0
RUL plafonnée maximale : 125
Tous les moteurs terminent à zéro : True


## 5. Sélection des variables

In [6]:
constant_columns = [
    "setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

excluded_columns = [
    "engine_id",
    "cycle",
    "max_cycle",
    "rul",
    "rul_capped",
    *constant_columns
]

feature_columns = [
    column
    for column in train.columns
    if column not in excluded_columns
]

useful_sensor_columns = [
    column
    for column in feature_columns
    if column.startswith("sensor_")
]

print("Variables initiales conservées :", len(feature_columns))
print("Capteurs utiles :", len(useful_sensor_columns))

Variables initiales conservées : 17
Capteurs utiles : 15


## 6. Création des caractéristiques temporelles

In [7]:
WINDOW_SIZE = 5


def add_temporal_features(df, sensor_columns, window_size):
    result = df.copy()

    for column in sensor_columns:
        result[f"{column}_ma{window_size}"] = (
            result.groupby("engine_id")[column]
            .transform(
                lambda values: values.rolling(
                    window=window_size,
                    min_periods=1
                ).mean()
            )
        )

        result[f"{column}_std{window_size}"] = (
            result.groupby("engine_id")[column]
            .transform(
                lambda values: values.rolling(
                    window=window_size,
                    min_periods=1
                ).std(ddof=0)
            )
        )

        result[f"{column}_delta{window_size}"] = (
            result.groupby("engine_id")[column]
            .diff(periods=window_size)
            .fillna(0)
        )

    return result


train = add_temporal_features(
    train,
    useful_sensor_columns,
    WINDOW_SIZE
)

test = add_temporal_features(
    test,
    useful_sensor_columns,
    WINDOW_SIZE
)

temporal_feature_columns = []

for sensor in useful_sensor_columns:
    temporal_feature_columns.extend([
        f"{sensor}_ma{WINDOW_SIZE}",
        f"{sensor}_std{WINDOW_SIZE}",
        f"{sensor}_delta{WINDOW_SIZE}"
    ])

final_feature_columns = (
    feature_columns
    + temporal_feature_columns
)

print("Nombre total de variables :", len(final_feature_columns))

print(
    "Valeurs manquantes train :",
    train[final_feature_columns].isna().sum().sum()
)

print(
    "Valeurs manquantes test :",
    test[final_feature_columns].isna().sum().sum()
)

Nombre total de variables : 62
Valeurs manquantes train : 0
Valeurs manquantes test : 0


## 7. Métriques d’évaluation

In [8]:
def nasa_score(y_true, y_pred):
    errors = (
        np.asarray(y_pred)
        - np.asarray(y_true)
    )

    penalties = np.where(
        errors < 0,
        np.exp(-errors / 13) - 1,
        np.exp(errors / 10) - 1
    )

    return penalties.sum()


def calculate_rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

## 8. Validation croisée par moteur

In [9]:
group_kfold = GroupKFold(n_splits=5)

folds = list(
    group_kfold.split(
        train,
        groups=train["engine_id"]
    )
)

rmse_folds = []
nasa_folds = []

predictions_oof = np.zeros(len(train))

for fold_number, (train_idx, val_idx) in enumerate(
    folds,
    start=1
):
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    X_train = train.iloc[
        train_idx
    ][final_feature_columns]

    y_train = train.iloc[
        train_idx
    ]["rul_capped"]

    X_val = train.iloc[
        val_idx
    ][final_feature_columns]

    y_val = train.iloc[
        val_idx
    ]["rul_capped"]

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    predictions_oof[val_idx] = predictions

    rmse_fold = calculate_rmse(
        y_val,
        predictions
    )

    nasa_fold = nasa_score(
        y_val,
        predictions
    )

    rmse_folds.append(rmse_fold)
    nasa_folds.append(nasa_fold)

    print(
        f"Fold {fold_number} — "
        f"RMSE : {rmse_fold:.2f} | "
        f"Score NASA : {nasa_fold:.2f}"
    )

y_oof = train["rul_capped"].to_numpy()

print(
    "\nRMSE moyenne :",
    round(np.mean(rmse_folds), 2)
)

print(
    "Score NASA moyen :",
    round(np.mean(nasa_folds), 2)
)

print(
    "RMSE OOF globale :",
    round(calculate_rmse(y_oof, predictions_oof), 2)
)

print(
    "Score NASA OOF global :",
    round(nasa_score(y_oof, predictions_oof), 2)
)

Fold 1 — RMSE : 20.19 | Score NASA : 93840.87
Fold 2 — RMSE : 19.00 | Score NASA : 47107.06
Fold 3 — RMSE : 18.12 | Score NASA : 61842.69
Fold 4 — RMSE : 17.93 | Score NASA : 54100.06
Fold 5 — RMSE : 15.84 | Score NASA : 26469.93

RMSE moyenne : 18.22
Score NASA moyen : 56672.12
RMSE OOF globale : 18.27
Score NASA OOF global : 283360.61


## 9. Calibration prudente des prédictions

In [10]:
CORRECTION_PRUDENTE = 11

predictions_oof_prudentes = np.clip(
    predictions_oof - CORRECTION_PRUDENTE,
    0,
    RUL_CAP
)

print(
    "RMSE OOF sans correction :",
    round(calculate_rmse(y_oof, predictions_oof), 2)
)

print(
    "RMSE OOF avec correction :",
    round(
        calculate_rmse(
            y_oof,
            predictions_oof_prudentes
        ),
        2
    )
)

print(
    "Score NASA OOF sans correction :",
    round(nasa_score(y_oof, predictions_oof), 2)
)

print(
    "Score NASA OOF avec correction :",
    round(
        nasa_score(
            y_oof,
            predictions_oof_prudentes
        ),
        2
    )
)

RMSE OOF sans correction : 18.27
RMSE OOF avec correction : 21.34
Score NASA OOF sans correction : 283360.61
Score NASA OOF avec correction : 173247.67


## 10. Entraînement final et évaluation sur le test officiel

In [11]:
final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    train[final_feature_columns],
    train["rul_capped"]
)

test_last_cycles = (
    test.sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

test_predictions = final_model.predict(
    test_last_cycles[final_feature_columns]
)

test_predictions_prudentes = np.clip(
    test_predictions - CORRECTION_PRUDENTE,
    0,
    RUL_CAP
)

y_test = rul_test["rul"].to_numpy()

rmse_test = calculate_rmse(
    y_test,
    test_predictions
)

nasa_test = nasa_score(
    y_test,
    test_predictions
)

rmse_test_prudent = calculate_rmse(
    y_test,
    test_predictions_prudentes
)

nasa_test_prudent = nasa_score(
    y_test,
    test_predictions_prudentes
)

print("Prédictions brutes")
print("RMSE :", round(rmse_test, 2))
print("Score NASA :", round(nasa_test, 2))

print("\nPrédictions prudentes")
print("RMSE :", round(rmse_test_prudent, 2))
print("Score NASA :", round(nasa_test_prudent, 2))

Prédictions brutes
RMSE : 18.97
Score NASA : 1053.9

Prédictions prudentes
RMSE : 20.27
Score NASA : 721.91


## 11. Résultats finaux

In [12]:
final_results = pd.DataFrame({
    "evaluation": [
        "Validation croisée — moyenne",
        "Validation OOF — brute",
        "Validation OOF — prudente",
        "Test officiel — brut",
        "Test officiel — prudent"
    ],
    "rmse": [
        np.mean(rmse_folds),
        calculate_rmse(y_oof, predictions_oof),
        calculate_rmse(y_oof, predictions_oof_prudentes),
        rmse_test,
        rmse_test_prudent
    ],
    "score_nasa": [
        np.mean(nasa_folds),
        nasa_score(y_oof, predictions_oof),
        nasa_score(y_oof, predictions_oof_prudentes),
        nasa_test,
        nasa_test_prudent
    ]
})

display(
    final_results.round(2)
)

,evaluation,rmse,score_nasa
0,Validation croisée — moyenne,18.22,56672.12
1,Validation OOF — brute,18.27,283360.61
2,Validation OOF — prudente,21.34,173247.67
3,Test officiel — brut,18.97,1053.90
4,Test officiel — prudent,20.27,721.91
